# FXMacroData × VectorBT — FX Strategy Backtesting

This notebook shows how to pull macro and FX data from the
[FXMacroData API](https://fxmacrodata.com) and backtest trading
strategies with [VectorBT](https://vectorbt.dev).

**USD announcements are public for the most recent 90 days; EUR/USD forex needs an API key**
- EUR/USD daily spot prices via `/v1/forex/eur/usd`
- USD policy rate via `/v1/announcements/usd/policy_rate`

**Professional API key access** (enter your key in the Configuration cell below)
- EUR policy rate + other protected non-USD announcement series
- Get a key at https://fxmacrodata.com/api-management

---

**Strategies covered**

| # | Strategy | Requires Pro key? |
|---|---|---|
| 1 | Moving-average crossover on EUR/USD | No |
| 2 | Vectorised parameter sweep (MA window grid search) | No |
| 3 | Interest-rate carry trade (rate differential signal) | Yes |
| 4 | Multi-currency carry basket | Yes |

## 1  Setup & imports

In [ ]:
# Install dependencies (uncomment if running for the first time)
# !pip install -r requirements.txt

In [ ]:
import datetime
import warnings
from typing import Optional

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import requests
import vectorbt as vbt

warnings.filterwarnings("ignore")
print(
    f"vectorbt {vbt.__version__}  |  pandas {pd.__version__}  |  numpy {np.__version__}"
)

## 2  Configuration

Set `API_KEY` to your Professional key to unlock EUR and multi-currency data.  
Leave it empty — all free-tier strategies will still run.

In [ ]:
API_BASE = "https://fxmacrodata.com/api/v1"
API_KEY: Optional[str] = ""  # ← paste your key here, or leave blank

# Date range for backtests — adjust as needed
END_DATE = datetime.date.today().isoformat()
START_DATE = (datetime.date.today() - datetime.timedelta(days=3 * 365)).isoformat()

# Initial cash (in quote-currency units, i.e. USD for EUR/USD)
INIT_CASH = 10_000.0

# ── helpers ────────────────────────────────────────────────────────────────────
_api_key = API_KEY.strip() or None
print(f"Date range : {START_DATE} → {END_DATE}")
print(f"API key    : {'set ✅' if _api_key else 'not set — free tier only'}")

## 3  Data helpers

Two thin wrappers around the FXMacroData REST API:

- **`fetch_forex(base, quote)`** → daily FX spot price as `pd.Series`
- **`fetch_indicator(currency, indicator)`** → macro indicator as `pd.Series`

In [ ]:
def fetch_forex(
    base: str,
    quote: str,
    api_key: Optional[str] = _api_key,
    start_date: str = START_DATE,
    end_date: str = END_DATE,
) -> pd.Series:
    """Return daily FX spot prices as a DatetimeIndex Series.

    FX spot rates require an API key.
    """
    params: dict = {"start_date": start_date, "end_date": end_date}
    if api_key:
        params["api_key"] = api_key
    resp = requests.get(
        f"{API_BASE}/forex/{base.lower()}/{quote.lower()}",
        params=params,
        timeout=20,
    )
    resp.raise_for_status()
    rows = resp.json().get("data", [])
    if not rows:
        return pd.Series(dtype=float, name=f"{base.upper()}/{quote.upper()}")
    df = pd.DataFrame(rows)
    df["date"] = pd.to_datetime(df["date"])
    return (
        df.set_index("date")["val"]
        .sort_index()
        .rename(f"{base.upper()}/{quote.upper()}")
    )


def fetch_indicator(
    currency: str,
    indicator: str,
    api_key: Optional[str] = _api_key,
    start_date: str = START_DATE,
    end_date: str = END_DATE,
) -> pd.Series:
    """Return a macro indicator as a DatetimeIndex Series.

    USD announcement data is public. Other currencies require a Professional API key.
    """
    params: dict = {"start_date": start_date, "end_date": end_date}
    if api_key:
        params["api_key"] = api_key
    resp = requests.get(
        f"{API_BASE}/announcements/{currency.lower()}/{indicator}",
        params=params,
        timeout=20,
    )
    if resp.status_code == 401:
        print(f"⚠️  {currency.upper()} {indicator}: API key required — skipping.")
        return pd.Series(dtype=float, name=f"{currency.upper()}_{indicator}")
    resp.raise_for_status()
    rows = resp.json().get("data", [])
    if not rows:
        return pd.Series(dtype=float, name=f"{currency.upper()}_{indicator}")
    df = pd.DataFrame(rows)
    df["date"] = pd.to_datetime(df["date"])
    return (
        df.set_index("date")["val"]
        .sort_index()
        .rename(f"{currency.upper()}_{indicator}")
    )


print("Helper functions defined ✅")

## 4  Fetch EUR/USD price data

In [ ]:
eurusd = fetch_forex("EUR", "USD")

print(f"EUR/USD: {len(eurusd)} daily observations")
print(f"  Range : {eurusd.index[0].date()} → {eurusd.index[-1].date()}")
print(f"  Latest: {eurusd.iloc[-1]:.4f}")
eurusd.tail()

In [ ]:
# Quick price chart
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=eurusd.index,
        y=eurusd.values,
        mode="lines",
        name="EUR/USD",
        line=dict(color="#1E88E5", width=1.5),
    )
)
fig.update_layout(
    title="EUR/USD Daily Spot Rate",
    xaxis_title="Date",
    yaxis_title="Price",
    template="plotly_white",
    height=350,
    margin=dict(l=40, r=20, t=50, b=40),
)
fig.show()

## 5  Strategy 1 — Moving-Average Crossover

Classic trend-following rule:  
- **Enter long** when the fast MA crosses *above* the slow MA.  
- **Exit** when the fast MA crosses *below* the slow MA.

Requires an API key — uses EUR/USD price only.

In [ ]:
FAST_WINDOW = 10  # days
SLOW_WINDOW = 50  # days

fast_ma = vbt.MA.run(eurusd, window=FAST_WINDOW, short_name="fast")
slow_ma = vbt.MA.run(eurusd, window=SLOW_WINDOW, short_name="slow")

entries = fast_ma.ma_crossed_above(slow_ma)
exits = fast_ma.ma_crossed_below(slow_ma)

pf_ma = vbt.Portfolio.from_signals(
    close=eurusd,
    entries=entries,
    exits=exits,
    freq="1D",
    init_cash=INIT_CASH,
    fees=0.0001,  # 1 pip spread approximation
    slippage=0.0001,
)

print(f"Strategy 1: {FAST_WINDOW}/{SLOW_WINDOW} MA Crossover — EUR/USD")
print("=" * 55)
print(pf_ma.stats())

In [ ]:
# Equity curve + drawdown
pf_ma.plot().show()

In [ ]:
# Overlay price with MA lines and trade markers
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=eurusd.index,
        y=eurusd.values,
        mode="lines",
        name="EUR/USD",
        opacity=0.6,
        line=dict(color="#90A4AE", width=1),
    )
)
fig.add_trace(
    go.Scatter(
        x=fast_ma.ma.index,
        y=fast_ma.ma.values,
        mode="lines",
        name=f"MA({FAST_WINDOW})",
        line=dict(color="#1E88E5", width=1.5),
    )
)
fig.add_trace(
    go.Scatter(
        x=slow_ma.ma.index,
        y=slow_ma.ma.values,
        mode="lines",
        name=f"MA({SLOW_WINDOW})",
        line=dict(color="#E53935", width=1.5),
    )
)

# Trade entry markers
entry_dates = eurusd.index[entries.values]
entry_prices = eurusd.loc[entry_dates]
exit_dates = eurusd.index[exits.values]
exit_prices = eurusd.loc[exit_dates]

fig.add_trace(
    go.Scatter(
        x=entry_dates,
        y=entry_prices.values,
        mode="markers",
        name="Entry",
        marker=dict(symbol="triangle-up", size=10, color="#43A047"),
    )
)
fig.add_trace(
    go.Scatter(
        x=exit_dates,
        y=exit_prices.values,
        mode="markers",
        name="Exit",
        marker=dict(symbol="triangle-down", size=10, color="#E53935"),
    )
)

fig.update_layout(
    title=f"EUR/USD — MA({FAST_WINDOW}/{SLOW_WINDOW}) Crossover Signals",
    xaxis_title="Date",
    yaxis_title="EUR/USD",
    template="plotly_white",
    height=420,
    margin=dict(l=40, r=20, t=50, b=40),
    legend=dict(orientation="h", y=1.02, x=1, xanchor="right"),
)
fig.show()

## 6  Strategy 2 — Parameter Optimisation (MA window grid search)

VectorBT's vectorised engine lets us test **hundreds of parameter
combinations in a single pass** — just pass arrays to `MA.run()`.  
Here we sweep fast windows `[5, 10, 15, 20]` × slow windows
`[30, 40, 50, 60, 80, 100]` and rank by Sharpe ratio.

In [ ]:
fast_windows = np.arange(5, 25, 5)  # [5, 10, 15, 20]
slow_windows = np.arange(30, 110, 10)  # [30, 40, 50, 60, 70, 80, 90, 100]

print(
    f"Testing {len(fast_windows)} × {len(slow_windows)} = "
    f"{len(fast_windows) * len(slow_windows)} parameter combinations…"
)

fast_ma_grid = vbt.MA.run(eurusd, window=fast_windows, short_name="fast")
slow_ma_grid = vbt.MA.run(eurusd, window=slow_windows, short_name="slow")

entries_grid = fast_ma_grid.ma_crossed_above(slow_ma_grid)
exits_grid = fast_ma_grid.ma_crossed_below(slow_ma_grid)

pf_grid = vbt.Portfolio.from_signals(
    close=eurusd,
    entries=entries_grid,
    exits=exits_grid,
    freq="1D",
    init_cash=INIT_CASH,
    fees=0.0001,
    slippage=0.0001,
)

print("Grid search complete ✅")

In [ ]:
# Collect key metrics for every parameter combination
sharpe = pf_grid.sharpe_ratio()
ret = pf_grid.total_return()
max_dd = pf_grid.max_drawdown()

results = pd.DataFrame(
    {
        "sharpe": sharpe,
        "total_return": ret,
        "max_drawdown": max_dd,
    }
).sort_values("sharpe", ascending=False)

print("Top 10 parameter combinations by Sharpe ratio:")
print(results.head(10).to_string())

In [ ]:
# Heatmap: Sharpe ratio by (fast_window, slow_window)
sharpe_df = sharpe.unstack(level="fast_window").loc[
    :, ~sharpe.unstack(level="fast_window").columns.duplicated()
]

fig = go.Figure(
    go.Heatmap(
        z=sharpe.unstack(level="fast_window").values,
        x=[str(w) for w in fast_windows],
        y=[str(w) for w in slow_windows],
        colorscale="RdYlGn",
        colorbar=dict(title="Sharpe"),
        text=np.round(sharpe.unstack(level="fast_window").values, 2),
        texttemplate="%{text}",
    )
)
fig.update_layout(
    title="EUR/USD MA Crossover — Sharpe Ratio Heatmap",
    xaxis_title="Fast window (days)",
    yaxis_title="Slow window (days)",
    template="plotly_white",
    height=420,
    margin=dict(l=60, r=20, t=50, b=50),
)
fig.show()

In [ ]:
# Best parameter combination
best_idx = results.index[0]
print(f"Best combination: fast={best_idx[0]} / slow={best_idx[1]}")
print(f"  Sharpe ratio : {results.loc[best_idx, 'sharpe']:.3f}")
print(f"  Total return : {results.loc[best_idx, 'total_return']:.1%}")
print(f"  Max drawdown : {results.loc[best_idx, 'max_drawdown']:.1%}")

## 7  Strategy 3 — Interest-Rate Carry Trade

A classic FX carry strategy: go **long EUR/USD** when the ECB deposit
rate is higher than the Fed Funds rate (positive EUR carry), and go
**flat** (or short) when USD yields are higher.

The rate differential is monthly (policy decisions); we **forward-fill**
it onto the daily price index before generating signals.

> **Requires a Professional API key** to fetch the EUR policy rate.  
> Set `API_KEY` in the Configuration cell and re-run this section.

In [ ]:
# Fetch USD and EUR policy rates
usd_rate = fetch_indicator("usd", "policy_rate")
eur_rate = fetch_indicator("eur", "policy_rate")  # requires Pro key

if usd_rate.empty:
    print("⚠️  No USD rate data — check START_DATE / END_DATE range.")
if eur_rate.empty:
    print("⚠️  No EUR rate data — set API_KEY to run the carry strategy.")
else:
    print(
        f"USD policy rate — latest: {usd_rate.iloc[-1]:.2f}% ({usd_rate.index[-1].date()})"
    )
    print(
        f"EUR policy rate — latest: {eur_rate.iloc[-1]:.2f}% ({eur_rate.index[-1].date()})"
    )

In [ ]:
if eur_rate.empty or usd_rate.empty:
    print("Skipping carry strategy — API key required for EUR policy rate.")
    print("Set API_KEY in the Configuration cell and re-run.")
else:
    # Align monthly policy rates onto the daily price index using forward-fill
    idx = eurusd.index
    usd_daily = usd_rate.reindex(idx).ffill().bfill()
    eur_daily = eur_rate.reindex(idx).ffill().bfill()

    # Rate differential: EUR − USD
    # Positive  → ECB rate > Fed rate → long EUR/USD (EUR carries more yield)
    # Negative  → Fed rate > ECB rate → flat (or short)
    rate_diff = eur_daily - usd_daily

    # Entry when differential turns positive; exit when it turns negative
    entries_carry = rate_diff.vbt.crossed_above(0)
    exits_carry = rate_diff.vbt.crossed_below(0)

    pf_carry = vbt.Portfolio.from_signals(
        close=eurusd,
        entries=entries_carry,
        exits=exits_carry,
        freq="1D",
        init_cash=INIT_CASH,
        fees=0.0001,
        slippage=0.0001,
    )

    print("Strategy 3: EUR/USD Interest-Rate Carry Trade")
    print("=" * 50)
    print(pf_carry.stats())

In [ ]:
if eur_rate.empty or usd_rate.empty:
    print("Skipping — set API_KEY above.")
else:
    # Plot rate differential over time
    fig = go.Figure()
    fig.add_trace(
        go.Bar(
            x=rate_diff.index,
            y=rate_diff.values,
            name="EUR−USD rate differential (%)",
            marker=dict(
                color=np.where(rate_diff.values >= 0, "#43A047", "#E53935"),
            ),
        )
    )
    fig.add_hline(y=0, line=dict(color="black", width=1, dash="dash"))
    fig.update_layout(
        title="EUR vs USD Policy Rate Differential",
        xaxis_title="Date",
        yaxis_title="EUR rate − USD rate (%)",
        template="plotly_white",
        height=350,
        margin=dict(l=50, r=20, t=50, b=40),
    )
    fig.show()

## 8  Strategy 4 — Multi-Currency Carry Basket

Extend the carry idea to multiple pairs simultaneously.  VectorBT
accepts a `pd.DataFrame` as `close`, so we can backtest all pairs
in a single call and compare them in one chart.

> **Requires a Professional API key.**

In [ ]:
# Pairs to include in the basket (all quoted vs USD)
BASKET_PAIRS: list[tuple[str, str]] = [
    ("EUR", "USD"),
    ("GBP", "USD"),
    ("AUD", "USD"),
    ("NZD", "USD"),
    ("CAD", "USD"),
]

if not _api_key:
    print("Skipping multi-currency basket — API key required.")
    print("Set API_KEY in the Configuration cell and re-run.")
else:
    basket_prices: dict[str, pd.Series] = {}
    basket_rates: dict[str, pd.Series] = {}

    for base, quote in BASKET_PAIRS:
        prices = fetch_forex(base, quote)
        rate = fetch_indicator(base.lower(), "policy_rate")
        if not prices.empty and not rate.empty:
            basket_prices[f"{base}/{quote}"] = prices
            basket_rates[base] = rate
        else:
            print(f"  Skipping {base}/{quote} — no data.")

    basket_rates["USD"] = fetch_indicator("usd", "policy_rate")

    print(f"\nPairs in basket: {list(basket_prices.keys())}")

In [ ]:
if not _api_key or not basket_prices:
    print("Skipping — set API_KEY above.")
else:
    # Build a common daily index across all pairs
    close_df = pd.concat(basket_prices.values(), axis=1, keys=basket_prices.keys())
    close_df = close_df.ffill().dropna(how="all")
    idx = close_df.index

    entries_basket_list: list[pd.Series] = []
    exits_basket_list: list[pd.Series] = []

    usd_daily_bk = basket_rates["USD"].reindex(idx).ffill().bfill()

    for pair_label, _ in basket_prices.items():
        base = pair_label.split("/")[0]
        base_rate_daily = basket_rates[base].reindex(idx).ffill().bfill()
        diff = base_rate_daily - usd_daily_bk

        entries_basket_list.append(diff.vbt.crossed_above(0).rename(pair_label))
        exits_basket_list.append(diff.vbt.crossed_below(0).rename(pair_label))

    entries_basket = pd.concat(entries_basket_list, axis=1)
    exits_basket = pd.concat(exits_basket_list, axis=1)

    pf_basket = vbt.Portfolio.from_signals(
        close=close_df,
        entries=entries_basket,
        exits=exits_basket,
        freq="1D",
        init_cash=INIT_CASH,
        fees=0.0001,
        slippage=0.0001,
    )

    print("Strategy 4: Multi-Currency Carry Basket")
    print("=" * 50)
    print(pf_basket.stats())

## 9  Strategy comparison

Plot the equity curves of all strategies that ran successfully on a
single chart so we can compare risk-adjusted performance at a glance.

In [ ]:
# Gather equity curves for strategies that completed
strategies: dict[str, vbt.Portfolio] = {}
strategies[f"MA({FAST_WINDOW}/{SLOW_WINDOW})"] = pf_ma

best_fast, best_slow = results.index[0][0], results.index[0][1]
fast_ma_best = vbt.MA.run(eurusd, window=best_fast, short_name="fast")
slow_ma_best = vbt.MA.run(eurusd, window=best_slow, short_name="slow")
entries_best = fast_ma_best.ma_crossed_above(slow_ma_best)
exits_best = fast_ma_best.ma_crossed_below(slow_ma_best)
pf_best = vbt.Portfolio.from_signals(
    close=eurusd,
    entries=entries_best,
    exits=exits_best,
    freq="1D",
    init_cash=INIT_CASH,
    fees=0.0001,
    slippage=0.0001,
)
strategies[f"MA({best_fast}/{best_slow}) [best]"] = pf_best

try:
    _ = pf_carry
    strategies["Carry Trade"] = pf_carry
except NameError:
    pass

# Plot equity curves
COLORS = ["#1E88E5", "#43A047", "#E53935", "#FB8C00", "#7B1FA2"]
fig = go.Figure()

for i, (label, pf) in enumerate(strategies.items()):
    equity = pf.value()
    fig.add_trace(
        go.Scatter(
            x=equity.index,
            y=equity.values,
            mode="lines",
            name=label,
            line=dict(color=COLORS[i % len(COLORS)], width=2),
        )
    )

fig.add_hline(
    y=INIT_CASH,
    line=dict(color="black", width=1, dash="dot"),
    annotation_text="Starting capital",
)
fig.update_layout(
    title="Strategy Equity Curves — EUR/USD",
    xaxis_title="Date",
    yaxis_title=f"Portfolio value (USD, starting {INIT_CASH:,.0f})",
    template="plotly_white",
    height=450,
    margin=dict(l=50, r=20, t=60, b=40),
    legend=dict(orientation="h", y=1.02, x=1, xanchor="right"),
)
fig.show()

In [ ]:
# Side-by-side summary table
rows = []
for label, pf in strategies.items():
    rows.append(
        {
            "Strategy": label,
            "Total Return": f"{pf.total_return():.1%}",
            "Sharpe": f"{pf.sharpe_ratio():.3f}",
            "Max Drawdown": f"{pf.max_drawdown():.1%}",
            "# Trades": int(pf.trades.count()),
            "Win Rate": (
                f"{pf.trades.win_rate():.1%}" if pf.trades.count() > 0 else "N/A"
            ),
        }
    )

summary = pd.DataFrame(rows).set_index("Strategy")
print("\nPerformance Summary")
print("=" * 65)
print(summary.to_string())

---

## Next steps

- **Add more signals** — use `fetch_indicator()` to pull inflation,
  PMI, or COT positioning data as additional filters.
- **Refine transaction costs** — adjust `fees` and `slippage` to
  match your broker's actual spreads.
- **Walk-forward validation** — split the data into in-sample
  optimisation and out-of-sample testing periods.
- **Live signals** — call the API in real time and emit entry/exit
  alerts to Slack, email, or a trading terminal.

### Useful links

- 🌐 [FXMacroData](https://fxmacrodata.com)
- 📖 [API Docs](https://fxmacrodata.com/documentation)
- 🔑 [Get an API key](https://fxmacrodata.com/api-management)
- 📦 [VectorBT docs](https://vectorbt.dev)
- 💬 [VectorBT discussions](https://github.com/polakowo/vectorbt/discussions)